# DocMind agentic RAG

This notebook documents the agentic retrieval-augmented generation (RAG) implementation in `app/agent/`. It explains *why* an agent is useful, the safety boundaries used by DocMind, the ReAct loop, tools, provider support, multi-hop retrieval, and practical testing. It is a design and operations notebook: execute cells from the repository root with the project virtual environment selected.

## 1. Classic RAG versus agentic RAG

Classic RAG is a fixed pipeline: `question → retrieve top-k → generate`. It is fast, predictable, and is still the right default for simple fact lookup. Its limitation is that the system always performs one retrieval pass with one query, even when the question is ambiguous, requires several documents, or has no local answer.

Agentic RAG introduces a bounded decision loop: `observe state → choose one tool → observe tool output → decide again`. The agent can search semantically, search exact terms, inspect a document, split a complex request, or stop. It is **not** allowed unlimited autonomy: a maximum iteration count, an allow-listed tool set, result limits, and an explicit web-search switch keep cost, latency, and risk bounded.

## 2. Architecture

```text
User / POST /query?mode=agentic
       │
       ▼
AgentOrchestrator ── short-term scratchpad + seen-action set
       │                       │
       │ plan JSON             └── observations retained only for this request
       ▼
AgentTools ── semantic_search (Chroma)
       ├────── keyword_search (BM25)
       ├────── read_document / read chunk
       ├────── web_search (optional, Wikipedia API)
       ├────── decompose
       ├────── clarify
       └────── synthesize
       │
       ▼
Grounded final answer + citations + agent_steps
```

The existing `KnowledgeBase` remains the source of truth. The agent does not talk directly to Chroma files or arbitrary shell commands. It uses the controlled Python tool wrappers in `app/agent/tools.py`.

## 3. ReAct theory and the DocMind variant

ReAct means **Reason + Act**. In the original pattern, an LLM alternates free-form thought, action, and observation. DocMind intentionally does **not** return hidden chain-of-thought. Instead, the planner returns a constrained JSON decision with a short operational reason:

```json
{
  "action": "keyword_search",
  "arguments": {"query": "Project Alpha owner", "top_k": 5},
  "reason": "Find the exact named entity in local documents."
}
```

The orchestration loop validates the action against the tool allow-list, executes it, saves the observation in a scratchpad, and asks for the next action. A malformed planner response safely falls back to a local search rather than executing arbitrary text. The loop terminates on `synthesize`, `clarify`, no evidence at the limit, or `AGENT_MAX_ITERATIONS`.

In [ ]:
from app.agent.tools import TOOLS
from pprint import pprint
pprint(TOOLS)


## 4. Tool contracts and safety

Each tool has a fixed name, description, and JSON input contract. This is the foundation of tool calling: the model selects only an allowed tool and provides typed arguments. `semantic_search` calls Chroma; `keyword_search` calls BM25; `read_document` reads only an already indexed document. Results are compacted before returning to the planner to control prompt size.

`web_search` is intentionally off by default (`DOCMIND_WEB_SEARCH_ENABLED=false`) and queries a fixed HTTPS Wikipedia API endpoint. It should never silently override local-document evidence. When enabled, its source is labeled so an answer can distinguish external information from DocMind documents. Production systems should add domain allow-lists, citation extraction from pages, request logging, timeouts, rate limits, and a user-visible external-search consent control.

In [ ]:
from app.backend import kb
from app.agent.tools import AgentTools

# This is a deterministic tool-level smoke test. Upload/ingest documents first,
# or run the ingestion cell below. It makes no LLM or web call.
tools = AgentTools(kb, web_enabled=False)
print(tools.keyword_search("Project Alpha owner", top_k=3))


## 5. Semantic versus keyword retrieval

BM25 is sparse lexical retrieval. It is particularly strong for exact names, product codes, dates, and legal language. Vector retrieval embeds query and chunks in the same semantic space and is useful for paraphrases such as *Who is accountable for Alpha?* instead of *Project Alpha owner*.

The agent may deliberately invoke both paths. This is more transparent than always calling the hybrid retriever: `agent_steps` lets you see what was tried. The classic `KnowledgeBase.retrieve()` remains hybrid BM25 + Chroma reciprocal-rank fusion (RRF), and is unchanged.

## 6. Multi-hop and decomposition

A multi-hop question needs multiple pieces of evidence. Example: *Compare the production dates for Project Alpha and Project Beta, then identify their owners.* The planner can emit:

```json
{"action":"decompose","arguments":{"questions":[
  "What is Project Alpha's production launch and owner?",
  "What is Project Beta's production launch and owner?"
]}}
```

The orchestrator puts these into a pending-question queue. It gathers evidence from subsequent retrieval calls, deduplicates it by stable ID/source, and passes the combined context to final synthesis. This is short-term task memory, not persistent user memory. It resets per request, which prevents one user’s temporary context influencing another request.

In [ ]:
# Optional: ingest the sample documents created for this repository.
# This can download/load the sentence-transformer model on first use.
from pathlib import Path
from app.backend import kb
for path in Path('docs').glob('*'):
    if path.is_file() and str(kb.upload_dir / path.name) not in kb.documents:
        kb.ingest(path.name, path.read_bytes())
print(f'{len(kb.documents)} documents, {kb.size} chunks indexed')


## 7. Providers: Claude and Ollama

The agent planner and final synthesizer use a provider-neutral plain-chat adapter. Set one provider before starting the API/kernel:

```env
DOCMIND_LLM_PROVIDER=claude
ANTHROPIC_API_KEY=...
ANTHROPIC_MODEL=claude-haiku-4-5-20251001

# OR local Ollama
DOCMIND_LLM_PROVIDER=ollama
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=llama3.2
```

Claude calls Anthropic Messages. Ollama calls its local `/api/chat` endpoint. Ollama models vary in JSON reliability; malformed planning JSON is handled by the safe local-search fallback. For a production agent, prefer native provider tool-calling schemas where available, validate arguments with Pydantic models, and record model/version with every request.

In [ ]:
# Requires configured Claude or a running Ollama server.
from dataclasses import asdict
from app.agent import answer_agentically

result = answer_agentically(
    "Compare Project Alpha and Project Beta production launch dates and owners.",
    kb,
    max_iterations=4,
)
data = asdict(result)
print(data['answer'])
print('\nSteps:')
for step in data['agent_steps']:
    print(step.get('action'), step.get('arguments'))


## 8. API and CLI usage

Classic mode remains the default and should be used for ordinary simple questions:

```powershell
python cli.py query "Who owns Project Alpha?" --mode classic
python cli.py query "Compare Alpha and Beta launch dates" --mode agentic --max-iterations 4
```

The REST endpoint is still `POST /query`; select the strategy in the request body:

```json
{"question":"Compare Alpha and Beta launch dates","mode":"agentic","top_k":5,"max_iterations":4}
```

The response contains `answer`, `citations`, `sources`, `retrieved`, `mode`, `clarification_needed`, and `agent_steps`. Do not expose full source text or operational traces to untrusted users without applying access control/redaction.

## 9. Testing strategy

Test three levels separately:

1. **Unit tests**: inject a fake knowledge base and fake planner/chat function. Assert tool validation, repeat-action suppression, max iteration behavior, clarification, and evidence deduplication without API calls.
2. **Integration tests**: ingest controlled documents, call `answer_agentically`, and assert the resulting sources/citations. Use a deterministic mock LLM plan to make this reproducible.
3. **Evaluation**: compare classic and agentic modes on the same frozen question set. Measure answer faithfulness, context quality, exact qrels retrieval metrics, cost, latency, iteration count, tool count, and clarification rate.

An agent is not automatically better. It should be retained only when its quality benefit exceeds its added latency/cost and it does not increase unsupported claims.

In [ ]:
# Basic project regression tests; no provider credentials or network are required.
!python -m pytest -q


## 10. Operational limits and next improvements

Current implementation limits: the scratchpad exists only for one request; web search is Wikipedia-only; planner decisions are JSON prompting rather than native function calls; and full-document reads are capped. Recommended next work: Pydantic argument validation per tool, session IDs backed by a secure store, document-level ACL filtering before retrieval, an evidence-sufficiency verifier, reranking, explicit source-type citations, per-tool time/token budgets, and agent-specific evaluation dashboards.

Never enable arbitrary URL fetching, code execution, shell tools, or unrestricted filesystem tools in a document agent without a threat model and strong sandboxing.